In [ ]:
def simulate_matches(initial_ratings: dict, k_factor: float, matches: list[tuple[str, str, float]]):
    """
    Simule une série de matchs et met à jour les évaluations Elo des équipes.

    :param initial_ratings: Dictionnaire {équipe: évaluation initiale}.
    :param k_factor: Le facteur K utilisé pour l'ajustement Elo.
    :param matches: Liste de tuples (Équipe A, Équipe B, Score réel A).
                      Score réel A doit être 1.0 (victoire), 0.5 (nul) ou 0.0 (défaite).
    """
    # Créer une copie mutable des ratings pour la simulation
    current_ratings = initial_ratings.copy()
    history = []

    print("--- Début de la Simulation ---")
    for i, match in enumerate(matches):
        team_a, team_b, score_actual_a = match
        rating_a = current_ratings[team_a]
        rating_b = current_ratings[team_b]

        # 1. Calculer les scores attendus
        e_a, e_b = calculate_expected_score(rating_a, rating_b)

        # 2. Mettre à jour les ratings
        new_rating_a, new_rating_b = update_elo(rating_a, rating_b, score_actual_a, k_factor)

        # 3. Mettre à jour l'état global et enregistrer l'historique
        current_ratings[team_a] = new_rating_a
        current_ratings[team_b] = new_rating_b

        history.append({
            "Match": f"{team_a} vs {team_b}",
            "Score Réel A": score_actual_a,
            "Attendu A": e_a,
            "Nouveau Elo A": round(new_rating_a, 2),
            "Nouveau Elo B": round(new_rating_b, 2)
        })

    print("\n--- Simulation Terminée ---")
    return history

# Exemple de simulation :
# Matchs: (Équipe A, Équipe B, Score réel A)
match_results = [
    ("Equipe A", "Equipe B", 1.0),  # Match 1: A gagne contre B
    ("Equipe B", "Equipe C", 0.5),  # Match 2: B fait match nul avec C
    ("Equipe A", "Equipe C", 0.0)   # Match 3: A perd contre C
]

simulation_history = simulate_matches(elo_ratings, k, match_results)

# Affichage des résultats dans un format lisible (DataFrame si pandas est disponible)
try:
    import pandas as pd
    df = pd.DataFrame(simulation_history)
    from IPython.display import display
    print("\nHistorique détaillé de la simulation:")
    display(df)
except ImportError:
    print("\n--- Historique détaillé (Pandas non disponible) ---")
    for record in simulation_history:
        print(f"Match: {record['Match']} | Score Réel A: {record['Score Réel A']:.1f} | Attendu A: {record['Attendu A']:.3f} | Nouveau Elo A: {record['Nouveau Elo A']} | Nouveau Elo B: {record['Nouveau Elo B']}")

## 4. Simuler des matchs successifs pour suivre l'évolution du classement

Pour simuler une série de matchs, nous allons maintenir un état actuel des évaluations Elo et itérer sur les résultats simulés. Nous utiliserons les fonctions `calculate_expected_score` et `update_elo` dans cette boucle de simulation.

In [ ]:
def update_elo(rating_a: float, rating_b: float, score_actual_a: float, k_factor: float) -> tuple[float, float]:
    """
    Calcule les nouvelles évaluations Elo pour A et B après un match.
    :param rating_a: Évaluation initiale de l'équipe A.
    :param rating_b: Évaluation initiale de l'équipe B.
    :param score_actual_a: Score réel de l'équipe A (1, 0.5 ou 0).
    :param k_factor: Le facteur K utilisé pour l'ajustement Elo.
    :return: Tuple des nouvelles évaluations (R'_A, R'_B).
    """
    # Calculer les nouveaux ratings
    new_rating_a = rating_a + k_factor * (score_actual_a - calculate_expected_score(rating_a, rating_b)[0])
    new_rating_b = rating_b + k_factor * ((1.0 - score_actual_a) - calculate_expected_score(rating_a, rating_b)[1])

    return new_rating_a, new_rating_b

# Exemple de test : A bat B (Score réel A=1)
r_a_new, r_b_new = update_elo(1500, 1500, 1.0, K_FACTOR)
print(f"Après victoire d'A contre B (1500 vs 1500): R'_A={r_a_new:.2f}, R'_B={r_b_new:.2f}")

# Exemple de test : A perd contre B (Score réel A=0)
r_a_new, r_b_new = update_elo(1500, 1500, 0.0, K_FACTOR)
print(f"Après défaite d'A contre B (1500 vs 1500): R'_A={r_a_new:.2f}, R'_B={r_b_new:.2f}")

## 3. Mettre à jour l'évaluation Elo après un match donné

La nouvelle évaluation ($R'_A$) est calculée en ajustant l'ancienne évaluation $R_A$ par la différence entre le score réel ($S_A$, où $S_A=1$ pour une victoire, $0.5$ pour un nul, et $0$ pour une défaite) et le score attendu ($E_A$), multipliée par le facteur K :
$$R'_A = R_A + K \times (S_A - E_A)$$

Nous allons créer une fonction qui prend les évaluations initiales, le score attendu et le résultat réel pour calculer la nouvelle évaluation.

In [ ]:
import numpy as np

def calculate_expected_score(rating_a: float, rating_b: float) -> tuple[float, float]:
    """
    Calcule le score attendu (probabilité de victoire) pour deux équipes.
    Retourne (E_A, E_B).
    :param rating_a: Évaluation Elo de l'équipe A.
    :param rating_b: Évaluation Elo de l'équipe B.
    :return: Tuple des scores attendus (probabilité de victoire pour A, probabilité de victoire pour B).
    """
    # Formule : E_A = 1 / (1 + 10^((R_B - R_A)/400))
    e_a = 1.0 / (1.0 + np.power(10, (rating_b - rating_a) / 400.0))
    # Puisque E_A + E_B = 1
    e_b = 1.0 - e_a
    return e_a, e_b

# Test de la fonction : A vs B (même niveau)
e_a_test, e_b_test = calculate_expected_score(1500, 1500)
print(f"Score attendu pour A vs B (1500 vs 1500): E_A={e_a_test:.4f}, E_B={e_b_test:.4f}")

# Test de la fonction : A beaucoup plus fort que B
e_a_strong, e_b_weak = calculate_expected_score(2000, 1000)
print(f"Score attendu pour A vs B (2000 vs 1000): E_A={e_a_strong:.4f}, E_B={e_b_weak:.4f}")

## 2. Calculer le score attendu entre deux équipes

Le score attendu ($E$) est la probabilité que l'équipe A batte l'équipe B, basée sur leur différence d'évaluation Elo. La formule utilisée est :
$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$
Où $R_A$ et $R_B$ sont les évaluations Elo de l'équipe A et B respectivement.

Nous allons implémenter une fonction pour calculer ce score attendu.

In [ ]:
# 1. Définir les constantes et paramètres du système Elo

def initialize_elo(initial_ratings: dict, k_factor: float = 32.0) -> tuple[dict, float]:
    """
    Initialise les évaluations Elo des équipes et retourne le facteur K.
    :param initial_ratings: Dictionnaire {équipe: évaluation initiale}.
    :param k_factor: Le facteur K utilisé pour l'ajustement Elo.
    :return: Tuple (dictionnaire d'évaluations, facteur K).
    """
    print("--- Initialisation des évaluations ---")
    # On suppose que le facteur K est constant pour ce calcul
    k = k_factor
    return initial_ratings, k

# Exemple d'utilisation : Définir les équipes et leurs évaluations initiales
teams = ["Equipe A", "Equipe B", "Equipe C"]
initial_ratings = {team: 1500 for team in teams} # Tous commencent à 1500 par défaut
K_FACTOR = 32.0

# Initialisation des données
elo_ratings, k = initialize_elo(initial_ratings, K_FACTOR)
print(f"Facteur K utilisé: {k}")
print(f"Évaluations initiales: {elo_ratings}")

# Simulation de l'Évaluation Elo d'une Équipe

Ce notebook est conçu pour calculer et simuler l'évolution du classement Elo d'une équipe donnée à travers une série de matchs. Il suit les étapes standard de la théorie Elo : définition des paramètres, calcul des scores attendus, mise à jour après un match, et simulation de plusieurs rencontres.